In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/03_silver_observations
# Purpose  : Split value_numeric from value_text based on TYPE. Used
#            to_timestamp instead of to_date to preserve time precision
#            (repeated measurements on the same day for the same patient).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(name="covid19_lakehouse.silver.observations", comment="Observations with value split by type")
@dlt.expect_or_drop("valid_patient_id", "patient_id IS NOT NULL")
def silver_observations():
    df = dlt.read_stream("covid19_lakehouse.bronze.observations")
    return (
        df.select(
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.to_timestamp("DATE").alias("observation_datetime"),
            F.col("CODE").alias("observation_code"),
            F.col("DESCRIPTION").alias("observation_description"),
            F.when(F.lower(F.col("TYPE")) == "numeric", F.col("VALUE").cast("double")).alias("value_numeric"),
            F.when(F.lower(F.col("TYPE")) == "text", F.col("VALUE")).alias("value_text"),
            F.col("UNITS").alias("units"),
            F.col("_ingested_at"),
        )
    )